# Exercises : Data Visualization in Python

This exercise aims to test your data visualization skills in Python

**Context**

To complete the exercises, you will have to download the **ADHD200** dataset. Both the phenotypic data and the resting-state fMRI data will be used.

**Instructions**

1. **Compliance**: Do not modify tje names of the output variables (e.g., q1_n_sujets) given in the commented lines.
2. **Execution**: Ensure that your code runs without errors from top to bottom.
3. **Parameters**: Strictly follow the parameters specified in each question.
4. **Data** : Only use the `pheno`, `func`, and `confounds` variables provided in the configuration cell (Section 0), which are already aligned with each other.

**Module validation**
To validate the module, please complete the following questions:
- Question 2: Relationship Between Age and Movement
- Question 3: Outliers and Participant Number
- Question 5: Visualize the Atlas
- Question 6: Time Series
  

The rest of the questions are bonus exercises.

## Section 0: Configuration and Data Loading

Execute these configuration cells. They download the data and prepare the variables you will use in the exercise. **Do not modify them.**

In [ ]:
# Configuration — do not modify
import warnings
warnings.filterwarnings('ignore')

import os
import statsmodels
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from cmcrameri import cm
from nilearn import datasets, plotting
from nilearn.image import math_img
from nilearn.maskers import NiftiLabelsMasker
from matplotlib.colors import ListedColormap
from nilearn.connectome import ConnectivityMeasure

print("Packages successfully imported.")

In [ ]:
# Loading ADHD200 datase — do not modify
data_dir = './nilearn_data'

adhd_dataset = datasets.fetch_adhd(n_subjects=40, data_dir=data_dir)

# Raw phenotypic data (might not cover all participants)
pheno_raw = adhd_dataset.phenotypic

print(f"Dataset loaded : {len(adhd_dataset.func)} functional images")
print(f"Raw phenotypic data : {pheno_raw.shape[0]} subjects")

In [ ]:
# Aligning data — do not modify
#
# Some subjects have fMRI data but no phenotypic data.
# We are keeping only subjects that have both.

func_ids = [int(os.path.basename(f).split('_')[0]) for f in adhd_dataset.func]
pheno_ids = set(pheno_raw['Subject'].values)
matched_idx = [i for i, fid in enumerate(func_ids) if fid in pheno_ids]
matched_fids = [func_ids[i] for i in matched_idx]

# Aligned data ready to use
pheno    = pheno_raw.set_index('Subject').loc[matched_fids].reset_index()
func     = [adhd_dataset.func[i] for i in matched_idx]
confounds = [adhd_dataset.confounds[i] for i in matched_idx]

print(f"Subjects with both imaging AND phenotypic data: {len(func)}")
print(f"Available phenotypic columns: {list(pheno.columns)}")

# Part 1 : Exploring phenotypic data

In this section, you will explore the clinical and demographic variables of the ADHD200 dataset.

### Question 1: Age Distribution by Sex

Visualize the `age` variable in the `pheno` dataframe based on the `sex` variable. Complete the `histplot` function below by specifying the correct variables. You must specify the following parameters:

- Specify the data structure (dataframe) to use in the function.
- The `age` variable must be on the x-axis.
- Age distributions must be separated by the `sex` variable. This must be specified within the same `histplot` function below.
- Overlay a Kernel Density Estimate (`kdeplot`) on the histogram.

You will need to specify the values for four parameters. **Leave the default values for other variables in the function.**

If needed, refer to the [`seaborn` documentation](https://seaborn.pydata.org/generated/seaborn.histplot.html).

In [ ]:
ax_q1 = plt.subplot()

sns.histplot(
    # ...
    ax=ax_q1
)

### Question 2: Relationship Between Age and Movement

Look at the relationship between the participant's age (`age`) and the movement levels (`MeanFD`) during the fMRI acquisition. Specifically, you will have to:

- Choose the appropriate function from the `seaborn` library to visualize only the relationship between the two variables, while including a regression model.
- Specify the data structure (dataframe) to use in the function.
- `age` should be treated as your independent variable.
- `MeanFD` should be treated as the dependent variable.

In [ ]:
ax_q2 = plt.subplot()

# Modify the lines below. Once you have replaced `<choose the right function>` with the correct function
# and replaced the `...` with the correct variables, uncomment the following lines:

#sns.<choose the right function>(
#    ...,
#    ax=ax_q2
#)

### Question 3: Outliers and Participant Number

While creating the figure showing the relationship between age and movement, you notice a participant with a potentially aberrant mean FD value. Extract the participant id for that participant directly from your figure.

Reproduce the figure from the previous question using `plotly.express`. For this question, the Plotly function is indicated. When you modify and run the cell below, you should be able to see the values for the variables `age`, `MeanFD`, and `Subject` when hovering your cursor over a point.

&#128161; To visualize the regression, use the Ordinary Least Squares (OLS) method.

In [ ]:
q3 = px.scatter(
    # ...
)

q3.show()

#### Question 3: Outliers and Participant Number (continued)

Now that you can access the participant number directly from your figure, store this number in the variable `q3_outlier` in the cell below.

In [ ]:
# REPLACE THE ... WITH YOUR ANSWER
q3_outlier = ...
print(f'Subject {q3_outlier} seems to have an aberrant mean FD value.')

### Question 4: Site Effect

Check the distribution of participants according to the acquisition site. Complete the cell below by adding the following parameters:
- Your figure size should have a width of 10 and a height of 3
- Visualize the sites on the x-axis
- The number of participants per site should be separated by group
- Change the y-axis name to the one stored in the variable `y_axis_name` in the cell below. &#128161; You will need to modify an attribute of the variable `ax_q4`.

Modify the `...` in the cell below as requested.

In [ ]:
# Specify the figure size
fig_q4, ax_q4 = plt.subplots(1, ...)

# Specify the requested parameters in the function below
sns.countplot(
    ...,
    ax=ax_q4
)

# Modify the axis name
y_axis_name = 'Number of participants' # DO NOT MODIFY
ax_q4. ...

# Part 2 : Exploring fMRI data

In this section, you will explore the functional fMRI data from the ADHD200 dataset.

In [ ]:
# Loading the BASC 12 ROIs atlas — do not modify
basc = datasets.fetch_atlas_basc_multiscale_2015(
    data_dir=data_dir, resolution=12
)
labels = basc.labels
atlas_img = basc.maps
print(f"Atlas loaded : {atlas_img}")
print(f"Number of regions: {len(labels)}")

### Question 5: Visualize the Atlas

Visualize the Regions of Interest (ROIs) using the `plot_roi` function from the `nilearn` library. Visualize the atlas according to the following instructions:

- Remove the cross showing the coordinate position for each slice.
- Add the title `Atlas BASC - 12 ROIs`
- Show the atlas at coordinates (0, 0, 0)

&#128161; Consult the documentation for the [`plot_roi`](https://nilearn.github.io/dev/modules/generated/nilearn.plotting.plot_roi.html) function to determine which parameters to include.

In [ ]:
title_q5 = 'Atlas BASC - 12 ROIs'

fig_q5 = plotting.plot_roi(
    ...
)

### Question 6: Time Series

Compare the temporal signals of **the first brain region** of the atlas between the subject with the lowest movement and the one with the highest movement (see figure generated in Question 3). Create a figure with 1 row and 2 columns. In the left column, visualize your ROI. In the right column, visualize the time series for both subjects.

**Left Column:**
- Visualize your ROI using `nilearn`'s `plot_roi` function
- Remove annotations
- Remove the colorbar
- Visualize only the axial slice according to the coordinates specified in the `coords` variable in the cell below
- Don't forget to specify your axis in the subplot!

&#128161; Consult the [`plot_roi`](https://nilearn.github.io/dev/modules/generated/nilearn.plotting.plot_roi.html) documentation for parameter details.

**Right column:**
- Visualize the time series of both subjects for your ROI using `matplotlib`'s `plot` function
- Specify the following labels for each subject: `'low'` for the subject with the least movement and `'high'` for the subject with the most movement
- Insert a legend at the bottom left of your subplot to indicate which color belongs to which subject
- The time series for the subject with the most movement (`'high'`) should be dashed (`'--'`)
- Add the title `'Volumes'` to your x-axis and set the font size to `14`
- Your line width should be `3`

&#128161; Refer to the visualization course tutorial and the [matplotlib documentation](https://matplotlib.org/stable/)if needed.

In [ ]:
# Create your figure layout
fig_q6, axes_q6 = plt.subplots(1, 2, figsize=(20, 5), width_ratios=[1, 3]) # DO NOT MODIFY

# Specify subject numbers
id_low_q6 = ... # TO MODIFY
id_high_q6 = ... # TO MODIFY

# Fetch data for the specified subjects
img_low_q6 = [f for f in func if str(id_low_q6) in f][0] # DO NOT MODIFY
img_high_q6 = [f for f in func if str(id_high_q6) in f][0] # DO NOT MODIFY

# Instantiate the masker
# DO NOT MODIFY
mask = NiftiLabelsMasker(
    labels_img=atlas_img,
    labels = labels
)
# Extract time series for the specified subjects
# DO NOT MODIFY
timeserie_low_q6=mask.fit_transform(
    img_low_q6
)
timeserie_high_q6=mask.fit_transform(
    img_high_q6
)

# Fetch time series for the first ROI
roi_id_q6 = ... # TO MODIFY
roi_low_q6 = timeserie_low_q6[:, roi_id_q6] # DO NOT MODIFY
roi_high_q6 = timeserie_high_q6[:, roi_id_q6] # DO NOT MODIFY

# Select specific region in the atlas
roi_atlas = math_img('img1==1', img1=atlas_img) # DO NOT MODIFY

# Add ROI to the figure
coords = (0,)

# TO MODIFY: SPECIFY REQUIRED PARAMETERS IN THE FUNCTION BELOW
plotting.plot_roi(
    ...
)

# Add time series to the figure
# TO MODIFY: SPECIFY AXIS, FUNCTION AND PARAMETERS FOR THE LOW MOVEMENT PARTICIPANT
axes_q6...
# TO MODIFY: SPECIFY AXIS, FUNCTION AND PARAMETERS FOR THE HIGH MOVEMENT PARTICIPANT
axes_q6...
# TO MODIFY: ADD LEGEND AND NECESSARY PARAMETERS
axes_q6...
# TO MODIFY: APPLY REQUESTED CHANGES TO THE X-AXIS
axes_q6...

### Question 7: Carpet plot

Compare the time series across all voxels for those two subjects. Use `nilearn`'s `plot_carpet` function. Specifically, you will need to:

- Create a figure with two rows (1 column)
- The top row should contain the carpet plot for the subject with the least movement (lowest Mean FD value)
- The bottom row should contain the carpet plot for the subject with the most movement (highest Mean FD value)
- For each subplot, add a title. For the top figure, use 'Participant with the lowest Mean FD value'. For the bottom figure, use 'Participant with the highest Mean FD value'
- Separate the carpet plot by atlas regions. **Hint:** you will need to use the `mask_img` and `mask_labels` parameters of the `plot_carpet` function

&#128161; The data to visualize has already been stored in variables in the previous question. To determine which variable to use for the mask, carefully check the expected variable type in the [nilearn documentation](https://nilearn.github.io/dev/modules/generated/nilearn.plotting.plot_carpet.html).


In [ ]:
labels_dict_q7 = {item: int(item) for item in labels if item != 'Background'} # DO NOT MODIFY

fig_q7, axes_q7 = plt.subplots(2, 1, figsize=(16, 12))

display_low_q7 = plotting.plot_carpet(
    ...
)
display_high_q7 = plotting.plot_carpet(
    ...
)